Forge dry-run on a free Kaggle GPU (LAW 34 second launcher).

This notebook checks out the estate repo at the pinned commit, then runs the UNCHANGED `forge/train.py` on the task + labelled dataset in this repo (internet is enabled). It writes `eval.json`; the launcher pulls `/kaggle/working/eval.json` back and files it through `forge/experiment_record.py` exactly like the Modal path. Nothing here invents a number.

In [ ]:
import json, os, subprocess, sys
os.makedirs('/kaggle/working/forge', exist_ok=True)
os.chdir('/kaggle/working/forge')

# Commit, task YAML and dataset are injected by the launcher via env (never hardcoded).
REPO = os.environ.get('FORGE_REPO', 'https://github.com/chidionyema/idp.git')
COMMIT = os.environ.get('FORGE_COMMIT', 'main')
TASK_REL = os.environ.get('FORGE_TASK', 'forge/tasks/ci-flake-triage.yaml')
DATA_REL = os.environ.get('FORGE_DATA', 'forge/datasets/ci-flake-triage.jsonl')
MAX_STEPS = os.environ.get('FORGE_MAX_STEPS', '-1')

print('checking out', REPO, 'at', COMMIT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', COMMIT, REPO, 'repo'], check=True)
os.chdir('repo')

# The portable trainer runs byte-unchanged on any CUDA box (LAW 34).
cmd = ['python', 'forge/train.py', '--task', TASK_REL, '--data', DATA_REL,
       '--max-steps', MAX_STEPS, '--out', '/kaggle/working/artifact']
print('running:', ' '.join(cmd))
r = subprocess.run(cmd)
print('TRAIN_RC', r.returncode)

# train.py writes eval.json into its --out dir; drop it at /kaggle/working/eval.json
# so `kaggle kernels output` pulls exactly one known file back.
ev_src = '/kaggle/working/artifact/eval.json'
if os.path.exists(ev_src):
    with open(ev_src) as f, open('/kaggle/working/eval.json', 'w') as g:
        g.write(f.read())
    print('EVAL_WRITTEN /kaggle/working/eval.json')
else:
    print('NO_EVAL train.py did not write eval.json; rc=', r.returncode)
    sys.exit(0 if r.returncode == 0 else 1)